# Multithreading & Multiprocessing in Python - Interview Guide

## 1. Difference between multithreading and multiprocessing in Python

**Key Differences:**
- **Memory Usage**: 
  - Multithreading: Threads share the same memory space
  - Multiprocessing: Each process has its own memory space
- **GIL Impact**:
  - Multithreading: Affected by GIL (only one thread executes Python bytecode at a time)
  - Multiprocessing: Not affected by GIL (true parallelism)
- **Creation Overhead**:
  - Threads are lighter and faster to create
  - Processes are heavier due to separate memory allocation
- **Use Cases**:
  - Multithreading: I/O-bound tasks
  - Multiprocessing: CPU-bound tasks

## 2. What is multithreading in Python, and how does it help?

Multithreading is a technique where multiple threads exist within a single process, sharing the same memory space but executing independently.

**Benefits:**
- Improved performance for I/O-bound applications (while one thread waits for I/O, others can execute)
- Better resource utilization
- Responsive UIs (main thread stays responsive while background threads handle tasks)
- Simplified modeling of concurrent operations

## 3. Common issues with multithreading in Python

1. **GIL Limitations**: Only one thread executes Python bytecode at a time
2. **Race Conditions**: When threads access shared data simultaneously
3. **Deadlocks**: Threads waiting indefinitely for resources held by each other
4. **Starvation**: Some threads may not get CPU time
5. **Thread Safety**: Built-in data structures aren't always thread-safe
6. **Debugging Difficulty**: Non-deterministic behavior makes bugs hard to reproduce

## 4. Difference between a thread and a process

| Characteristic | Thread | Process |
|---------------|--------|---------|
| Memory | Shares memory space | Has separate memory space |
| Creation | Lightweight | Heavyweight |
| Communication | Direct (shared memory) | Requires IPC (pipes, queues, etc.) |
| Crash Impact | Affects all threads in process | Only affects that process |
| Context Switching | Faster | Slower |
| Parallelism | Limited by GIL | True parallelism |

## 5. Python's Global Interpreter Lock (GIL) impact on multithreading

**GIL** is a mutex that protects access to Python objects, preventing multiple threads from executing Python bytecode simultaneously.

**Impacts:**
- **Limits Parallelism**: Only one thread executes Python code at a time
- **I/O-bound Benefits**: Threads can still be useful when waiting for I/O
- **CPU-bound Penalty**: No performance gain for CPU-intensive tasks
- **Workarounds**: Use multiprocessing or C extensions that release GIL

## 6. Using Python's threading module with example

```python
import threading
import time

def print_numbers():
    for i in range(5):
        time.sleep(1)
        print(i)

def print_letters():
    for letter in 'ABCDE':
        time.sleep(1.5)
        print(letter)

# Create threads
t1 = threading.Thread(target=print_numbers)
t2 = threading.Thread(target=print_letters)

# Start threads
t1.start()
t2.start()

# Wait for both threads to complete
t1.join()
t2.join()

print("Done!")
```

## 7. Thread synchronization and its importance

**Thread synchronization** coordinates the execution of threads to ensure controlled access to shared resources.

**Why important:**
- Prevents race conditions
- Ensures data consistency
- Avoids deadlocks
- Maintains proper execution order

**Common synchronization primitives:**
- Locks
- RLocks
- Semaphores
- Conditions
- Events

## 8. Purpose of the join() method in Python threading

The `join()` method blocks the calling thread (usually main thread) until the thread whose `join()` method is called terminates.

**Key points:**
- Ensures thread completion before proceeding
- Can specify timeout (how long to wait)
- Without join(), main thread may exit before threads complete
- Multiple joins can be used to wait for several threads

## 9. Implementing a thread-safe counter in Python

```python
import threading

class ThreadSafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()
    
    def increment(self):
        with self._lock:
            self._value += 1
    
    def value(self):
        with self._lock:
            return self._value

# Usage
counter = ThreadSafeCounter()

def worker():
    for _ in range(1000):
        counter.increment()

threads = []
for _ in range(10):
    t = threading.Thread(target=worker)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(counter.value())  # Should be 10000
```

## 10. Implementing multithreading in Python (example)

```python
import threading
import requests

def download_file(url, filename):
    print(f"Downloading {url}")
    response = requests.get(url)
    with open(filename, 'wb') as f:
        f.write(response.content)
    print(f"Finished {filename}")

urls = [
    ('https://example.com/file1.zip', 'file1.zip'),
    ('https://example.com/file2.zip', 'file2.zip'),
    ('https://example.com/file3.zip', 'file3.zip')
]

threads = []
for url, filename in urls:
    thread = threading.Thread(target=download_file, args=(url, filename))
    thread.start()
    threads.append(thread)

# Wait for all threads to complete
for thread in threads:
    thread.join()

print("All downloads completed!")
```

## 11. GIL (Global Interpreter Lock) and its significance

**GIL** is a mutex that allows only one thread to execute Python bytecode at a time in a single process.

**Significance:**
- Simplifies memory management (reference counting is thread-safe)
- Makes CPython implementation easier
- Impacts performance for CPU-bound multithreaded programs
- Doesn't affect I/O-bound operations much
- Only affects CPython (not Jython or IronPython)

## 12. Multiprocessing in Python vs multithreading

**Multiprocessing:**
- Uses separate memory spaces (processes)
- Not affected by GIL
- Better for CPU-bound tasks
- Higher overhead for process creation
- Requires inter-process communication (IPC)

**Multithreading:**
- Shares memory space
- Affected by GIL
- Better for I/O-bound tasks
- Lower overhead
- Easier communication between threads

## 13. Advantages of multiprocessing over threading

1. **True Parallelism**: Bypasses GIL limitations
2. **Better CPU Utilization**: For CPU-bound tasks
3. **Memory Isolation**: Processes don't share memory (more secure)
4. **Crash Resilience**: One process crashing doesn't affect others
5. **Scalability**: Can leverage multiple CPU cores effectively
6. **No Thread-Safety Concerns**: For non-shared data

## 14. Pool of workers in Python's multiprocessing

A **worker pool** is a group of processes that are created once and reused to execute multiple tasks.

**How it works:**
1. Create a fixed number of worker processes
2. Submit tasks to the pool
3. Pool distributes tasks to available workers
4. Results are collected as they complete

**Example:**
```python
from multiprocessing import Pool

def square(x):
    return x * x

if __name__ == '__main__':
    with Pool(4) as p:  # 4 worker processes
        results = p.map(square, range(10))
    print(results)  # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
```

## 15. Sharing data between processes in Python

1. **Shared Memory**:
   - `Value`, `Array` from multiprocessing module
   - `multiprocessing.shared_memory` (Python 3.8+)

2. **Manager Objects**:
   - `multiprocessing.Manager()` creates server process
   - Can share lists, dicts, queues, etc.

3. **Queues**:
   - `multiprocessing.Queue`
   - Thread-safe communication

4. **Pipes**:
   - `multiprocessing.Pipe()`
   - Two-way communication

**Example with shared memory:**
```python
from multiprocessing import Process, Value, Array

def f(n, a):
    n.value = 3.1415927
    for i in range(len(a)):
        a[i] = -a[i]

if __name__ == '__main__':
    num = Value('d', 0.0)
    arr = Array('i', range(10))

    p = Process(target=f, args=(num, arr))
    p.start()
    p.join()

    print(num.value)  # 3.1415927
    print(arr[:])     # [0, -1, -2, -3, -4, -5, -6, -7, -8, -9]
```

## 16. Difference between Queue and Pipe in multiprocessing

| Feature | Queue | Pipe |
|---------|-------|------|
| Connection | Multiple producers/consumers | Two endpoints only |
| Duplex | Typically one-way | Can be two-way |
| Complexity | Higher-level, more features | Lower-level, simpler |
| Performance | Slightly slower due to locking | Faster |
| Use Case | When multiple processes need to communicate | When exactly two processes communicate |

## 17. Managing race conditions in multiprocessing

1. **Locks**:
   - `multiprocessing.Lock()`
   - Ensures only one process accesses critical section

2. **RLocks**:
   - Reentrant locks (same process can acquire multiple times)

3. **Semaphores**:
   - Limit number of processes accessing resource

4. **Queues**:
   - Thread-safe data exchange

5. **Manager Objects**:
   - Provide synchronized data structures

**Example with Lock:**
```python
from multiprocessing import Process, Lock, Value

def increment(lock, counter):
    for _ in range(1000):
        with lock:
            counter.value += 1

if __name__ == '__main__':
    lock = Lock()
    counter = Value('i', 0)
    processes = [Process(target=increment, args=(lock, counter)) 
                for _ in range(10)]

    for p in processes:
        p.start()
    for p in processes:
        p.join()

    print(counter.value)  # Should be 10000
```

## 18. Use cases for multiprocessing over multithreading

1. **CPU-bound tasks**:
   - Mathematical computations
   - Data processing/transformation
   - Image/video processing

2. **When GIL is a bottleneck**:
   - Heavy Python bytecode execution

3. **Process isolation needed**:
   - When crashes shouldn't affect other workers
   - Security requirements

4. **Utilizing multiple cores**:
   - For true parallel execution

5. **Batch processing**:
   - Independent tasks that can be parallelized

## 19. Garbage collection in Python

**Garbage Collection (GC)** is automatic memory management that reclaims memory occupied by objects no longer in use.

**How it works in Python:**
1. **Reference Counting**:
   - Primary mechanism
   - Each object has count of references to it
   - When count reaches zero, memory is freed immediately

2. **Generational GC**:
   - Secondary mechanism for reference cycles
   - Three generations (new, middle, old)
   - Objects that survive collections move to older generations
   - Runs periodically (threshold-based)

**Key Aspects:**
- Mostly automatic
- Can be controlled with `gc` module
- `__del__` method for cleanup
- Can be disabled (but not recommended)